# Práctica evaluable S3–S4 (Regresiones)

## Dos problemas, dos formas de modelizar

En esta práctica aplicarás regresión lineal a la demanda horaria de bicicletas y regresión logística a una clasificación didáctica basada en la progresión continua del conjunto Diabetes de scikit-learn.

El objetivo no es producir muchos gráficos o modelos, sino justificar las decisiones y explicar qué muestran los resultados.

> **Importante:** Diabetes no indica quién tiene diabetes. `target` mide progresión observada un año después. La clase que se construirá es un corte didáctico de esa medida, no un diagnóstico ni una estimación clínica de riesgo.

Las instrucciones de cada apartado indican qué resultado hay que entregar y dónde escribir su interpretación.

## Cómo trabajar, entregar y consultar los materiales

1. Ejecuta las celdas existentes en orden. Cuando se pida código, añade una celda Python justo debajo y sigue las pistas; no borres las preguntas.
2. Completa las celdas **RESPUESTA** asociadas a las preguntas. No se solicita un informe final adicional.
3. En cada respuesta, explica qué hiciste, con qué datos, qué observas y una limitación. Usa cifras con unidades o denominadores cuando corresponda.
4. Mantén separados ajuste, validación y test. El test se reserva para la evaluación final, después de fijar el modelo y las decisiones.
5. **No cambies las partes ya preparadas.** En las celdas de código, las líneas marcadas `YA PREPARADO — NO MODIFICAR` se ejecutan tal como están. Completa únicamente los apartados `TODO` y los huecos indicados; si una celda mezcla ambas cosas, conserva las líneas preparadas y añade tu trabajo donde se pide.

### Mapa de consulta — vuelve a estos apartados cuando te atasques

- **Describir variables, histogramas, cajas y correlaciones:** [S1, descriptiva univariante y bivariante](sesion1/s1.ipynb); también [S3, inspección y exploración gráfica (apartados 3.1–3.2)](sesion3/s3.ipynb).
- **Intervalo y contraste de dos medias** (ampliación opcional): [S2.2, comparación de medias independientes](sesion2/s2_2.ipynb) y [S2.1, intervalos y contrastes](sesion2/s2_1.ipynb).
- **OLS simple/múltiple, residuos, selección AIC y evaluación de regresión:** [S3, apartados 3.3–3.14](sesion3/s3.ipynb).
- **Indicadores y categorías de referencia; regresión logística; OR; ROC-AUC; umbral y matriz de confusión:** [S4, apartados 4.2–4.6 y 4.7–4.10](sesion4/s4.ipynb).
- **Diabetes y sus variables:** [documentación de `load_diabetes`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html).

### Pasos nuevos guiados en esta práctica

S3/S4 no enseñan una partición cronológica interna para Bike. La celda de preparación separa automáticamente los días completos: reserva el 20 % final de las fechas como test externo y, dentro del bloque de entrenamiento, el 20 % final como validación. También deja preparado el diseño con indicadores de hora y alinea sus columnas entre ajuste, validación y test. **Estas líneas están resueltas y no deben modificarse.**

La tarea se centra en practicar código frecuente: descriptivas y gráficos básicos, llamadas a `OLS`/`GLM`, predicciones, AIC, RMSE, ROC-AUC y matriz de confusión. Se proporcionan los pasos más mecánicos de partición y codificación para que puedas concentrarte en escribir esas operaciones, comprobar sus resultados e interpretarlos. Bike usa una sola ventana temporal de validación, no una validación cruzada.

Para una predicción futura, `temp`, `hum` y `windspeed` solo estarán disponibles si se cuenta con pronósticos meteorológicos para esas horas. Si no, interpreta el análisis como predicción condicionada a los valores meteorológicos observados, no como una previsión operativa completa.

En Diabetes, se proporciona el protocolo de partición y se deja practicar el ajuste de modelos, la comparación por AUC y la evaluación con un umbral fijo de 0,5. **La partición, el cálculo de las etiquetas y el umbral ya están preparados; no los cambies.** No hace falta buscar ni optimizar umbrales.

### Fuentes de datos

- Bike Sharing, registros horarios: [repositorio UCI](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), archivo `hour.csv` (la plantilla lo descarga).
- Diabetes: [documentación de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html) (la plantilla lo carga).

La descarga de Bike requiere conexión a internet la primera vez. Si falla, avisame al correo (guille.rochinaaguas@gmail.com) o en clase; no sustituyas el conjunto por otro sin indicación.

## 1. Definir qué se va a estudiar

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;">
<strong>PREGUNTA 0</strong><br>
Antes de programar, completa la tabla con una frase por casilla. Para cada conjunto, indica la unidad de observación, qué mide la respuesta y su tipo, si el objetivo es descriptivo o predictivo, y una hipótesis de asociación sin afirmar causalidad.
</div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong></div>

| Caso | Unidad de observación | Respuesta y tipo | Enfoque (descriptivo/predictivo) | Hipótesis inicial (sin causalidad) |
|---|---|---|---|---|
| Bike Sharing |  |  |  |  |
| Diabetes |  |  |  |  |

**Dónde mirar:** [S3, introducción y modelo lineal (3.3)](sesion3/s3.ipynb) repasa respuesta continua y regresión; [S4, introducción](sesion4/s4.ipynb) contrasta regresión logística para respuestas binarias. Para el significado de `target`, consulta la [documentación de Diabetes](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html).

## 2. Preparar herramientas y cargar Bike Sharing

Las dos primeras celdas de código **ya están completas**: importan las bibliotecas, fijan la semilla y cargan los datos. Ejecútalas sin modificarlas. La tercera celda muestra un ejemplo parcial de inspección.

**Completa únicamente los apartados `TODO` del bloque 2.3**, siguiendo este orden:

1. Dimensiones y primeras filas (ya están hechas; no hace falta repetirlas).
2. Nombres de columnas y tipos de datos.
3. Valores ausentes y filas duplicadas.
4. Rango de fechas y orden de las observaciones.

**Dónde mirar:** [S3, apartado 3.1, inspección inicial](sesion3/s3.ipynb) muestra cómo revisar dimensiones, filas, tipos y resumen numérico. Aquí aplica esos mismos comandos a `bike`.

Anota en Markdown si encontraste problemas y qué decisión tomaste. En este conjunto, `cnt` es la demanda total por hora. **No uses `casual` ni `registered` como predictoras**, porque son componentes de `cnt`. `instant` es un índice y `dteday` se usará para ordenar, no como predictor numérico crudo.

In [ ]:
import io
import subprocess
import zipfile
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm  # OLS: S3 §3.3–3.4; GLM binomial: S4 §4.3.
from scipy import stats  # Contraste de dos medias opcional: S2.2, muestras independientes.
from sklearn.model_selection import train_test_split  # Particiones de Diabetes; protocolo de evaluación en S3 §3.13 y S4 §4.9.

# Métricas de regresión: S3 §3.14. Métricas de clasificación: S4 §§4.7–4.10.
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
)

SEMILLA = 42
print(f"Semilla para particiones aleatorias: {SEMILLA}")

In [ ]:
BIKE_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

# Descarga UCI con validación TLS normal; curl sirve de alternativa si el almacén
# de certificados del entorno causa problemas.
try:
    with urlopen(BIKE_URL, timeout=30) as respuesta:
        contenido_zip = respuesta.read()
except Exception:
    respuesta_curl = subprocess.run(
        ["curl", "--fail", "--location", "--silent", "--show-error", BIKE_URL],
        check=True,
        capture_output=True,
        timeout=60,
    )
    contenido_zip = respuesta_curl.stdout

with zipfile.ZipFile(io.BytesIO(contenido_zip)) as archivo_zip:
    ruta_hour = next(nombre for nombre in archivo_zip.namelist() if nombre.endswith("hour.csv"))
    with archivo_zip.open(ruta_hour) as archivo_csv:
        bike = pd.read_csv(archivo_csv, parse_dates=["dteday"])

print(f"Bike Sharing: {bike.shape[0]:,} filas × {bike.shape[1]} columnas")
display(bike.head())

In [ ]:
# 2.3 — Inspección inicial de Bike
# Referencia: S3 §3.1, inspección inicial de datos.
# YA PREPARADO: dimensiones y primeras filas; no hace falta cambiar esas líneas.
# Completa solo las instrucciones TODO de columnas, calidad y fechas.

# 1. Dimensiones y primeras filas
print("Dimensiones:", bike.shape)
display(bike.head())
# TODO: muestra nombres de columnas y tipos (pista: bike.columns y bike.dtypes).



# 2. Calidad básica
# TODO: calcula ausentes por columna y filas duplicadas.
# Pistas: isna().sum() y duplicated().sum().



# 3. Fechas y orden temporal
# TODO: muestra fecha mínima y máxima de dteday.


# TODO: comprueba orden por fecha y hora con sort_values(["dteday", "hr"]).


# No cambies el orden aquí; lo harás al crear la partición.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 1 — Inspección de Bike</strong><br>¿Qué representa una fila y qué mide <code>cnt</code>?<br><em>Pista: cada registro corresponde a una unidad de tiempo; expresa cnt con su unidad.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2 — Calidad e intervalo temporal</strong><br>¿Qué tamaño y periodo cubre el conjunto? ¿Qué comprobaciones de calidad hiciste y qué dicen?<br><em>Pista: cita el número de filas/columnas, ausentes, duplicados, fechas mínima y máxima, y (opcional/adicional) si el orden por fecha-hora ya se cumple.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3 — Decisión de limpieza y límite</strong><br>¿Qué decisión de limpieza tomaste y qué problema de calidad no puedes descartar?<br><em>Pista: si no eliminaste filas, justifícalo con las comprobaciones; menciona errores de medición que no se detectan contando ausentes y duplicados.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Explica si limpiaste o no y posibles limitaciones de las comprobaciones permanece.

</div>

## 3. Caso 1 — Bike: separar y describir

### 3.1. Reservar test y validación antes de explorar

Ordena las filas por fecha y hora. Reserva el **20 % final de las fechas completas** como test externo. Dentro del resto, reserva también el **20 % final de las fechas completas** como validación; el tramo inicial será el bloque de ajuste. No barajes las filas ni partas un mismo día entre bloques.

Usa el bloque de **ajuste** para describir los datos, ajustar los modelos y revisar sus residuos. La validación se reserva para comparar candidatos y elegir; el test, para una evaluación final. No uses ninguno de los dos bloques reservados para decidir transformaciones o especificaciones durante la exploración.

Al preparar las predictoras, excluye `cnt`, `casual`, `registered`, `instant` y `dteday`. Mantén `hr`, `season` y `workingday` disponibles como categorías cuando corresponda.

**Ya está preparado en el bloque de código siguiente — no lo cambies:** orden temporal, particiones por fechas completas, comprobaciones de separación y extracción de predictoras/respuesta para ajuste y validación. Ejecuta esas líneas tal como están. Completa solo la parte de descriptiva marcada `TODO`.

### 3.2. Descripción del bloque de ajuste

**Completa el bloque de código 3.1–3.2 que aparece a continuación**, usando solo `bike_ajuste`, para producir:

1. Una tabla de `cnt`, `temp`, `atemp`, `hum` y `windspeed` con tamaño, media, mediana, desviación típica, cuartiles, mínimo y máximo.
2. Un histograma de `cnt`.
3. Diagramas de caja de `cnt` por `season` y `workingday`, acompañados por el tamaño de cada grupo.
4. Un gráfico de la demanda media por hora (`hr`).
5. Una matriz de correlación de `cnt` y las principales variables numéricas.

**Dónde mirar:** para el resumen, los histogramas, boxplots y correlación, consulta [S1, descriptiva univariante y bivariante](sesion1/s1.ipynb). Para revisar tipos y exploración gráfica, consulta [S3, apartados 3.1–3.2](sesion3/s3.ipynb).

**Guía para escribir la respuesta:** elige dos patrones y cita el resultado concreto que los respalda; compara media y mediana de `cnt`; menciona una asociación o posible redundancia entre predictores; termina con una limitación. `cnt` es un recuento discreto, aunque lo trataremos numéricamente con OLS. No interpretes correlación como causalidad.

In [ ]:
# 3.1 — Partición cronológica por fechas completas
# YA PREPARADO — NO MODIFICAR: orden, particiones por día y comprobaciones.
# Referencia de protocolo: sesion3/s3.ipynb, celdas 125–130 (reservar y evaluar con test).
# La separación de validación por fechas completas es una adaptación específica de esta práctica.
# Ordeno los días, reservo los días más recientes para test y,
# dentro del resto, reservo los días finales para validación; nunca reparto horas al azar.
bike_ordenado = bike.sort_values(["dteday", "hr"]).copy()
fechas = bike_ordenado["dteday"].drop_duplicates().reset_index(drop=True)

# Reserva el 20% final de los días para el test externo.
n_test_fechas = max(1, int(np.ceil(0.20 * len(fechas))))
fecha_inicio_test = fechas.iloc[-n_test_fechas]
bike_train = bike_ordenado.loc[bike_ordenado["dteday"] < fecha_inicio_test].copy()
bike_test = bike_ordenado.loc[bike_ordenado["dteday"] >= fecha_inicio_test].copy()

# Reserva el 20% final de los días de entrenamiento para validación.
fechas_train = bike_train["dteday"].drop_duplicates().reset_index(drop=True)
n_validacion_fechas = max(1, int(np.ceil(0.20 * len(fechas_train))))
fecha_inicio_validacion = fechas_train.iloc[-n_validacion_fechas]
bike_ajuste = bike_train.loc[bike_train["dteday"] < fecha_inicio_validacion].copy()
bike_validacion = bike_train.loc[bike_train["dteday"] >= fecha_inicio_validacion].copy()

# Comprueba tamaños, rangos y que los bloques no compartan fechas.
for nombre, bloque in (("ajuste", bike_ajuste), ("validación", bike_validacion),
                       ("entrenamiento externo", bike_train), ("test", bike_test)):
    print(f"{nombre}: {len(bloque)} filas; {bloque['dteday'].min().date()} a {bloque['dteday'].max().date()}")
assert bike_ajuste["dteday"].max() < bike_validacion["dteday"].min()
assert bike_validacion["dteday"].max() < bike_test["dteday"].min()

# Prepara los bloques sin la respuesta, sus componentes ni identificadores temporales.
columnas_excluir = ["cnt", "casual", "registered", "instant", "dteday"]
predictores_bike = [columna for columna in bike.columns if columna not in columnas_excluir]
X_ajuste = bike_ajuste[predictores_bike].copy()
y_ajuste = bike_ajuste["cnt"].copy()
X_validacion = bike_validacion[predictores_bike].copy()
y_validacion = bike_validacion["cnt"].copy()

# 3.2 — Completa solo los TODO; usa exclusivamente bike_ajuste.
columnas_resumen = ["cnt", "temp", "atemp", "hum", "windspeed"]
# Referencias de técnicas: sesion1/s1.ipynb, celdas 132–156 (resumen, caja e histograma)
# y celdas 180–196 (correlación y dispersión); sesion3/s3.ipynb, celdas 8–14.
# TODO: usa bike_ajuste, describe().T y añade la mediana de cada variable.


# TODO: dibuja histograma de cnt (pista: sns.histplot).


# TODO: compara cnt por season y workingday con boxplots; informa también los tamaños.


# TODO: agrupa cnt por hr y representa la media (pista: groupby(...) ["cnt"].mean()).


# TODO: muestra la matriz de correlación de cnt y las variables numéricas principales.


# Interpreta patrones, no causalidad; revisa las preguntas que siguen.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 1 — Descripción de Bike</strong><br>¿Qué dos patrones de demanda destacarías por hora o estación? Incluye cifras y, si comparas grupos, sus tamaños.<br><em>Pista: usa la media agrupada por hora, las medianas de los boxplots y los recuentos por season/workingday.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Describe dos patrones y cita la cifra/grupo que los muestra.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2 — Forma de la demanda</strong><br>¿Cómo se comparan la media y mediana de <code>cnt</code> y qué sugiere la diferencia sobre su distribución?<br><em>Pista: toma ambas cifras de la tabla y relaciónalas con la cola del histograma.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3 — Asociaciones y límites</strong><br>¿Qué asociación lineal o redundancia destaca en la matriz? ¿Qué no permite concluir la correlación?<br><em>Pista: busca un par de predictores con coeficiente cercano a 1 y recuerda que la correlación solo resume asociación lineal, no causalidad.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

### Ampliación opcional — comparar días laborables y no laborables

#### Esta ampliación no es necesaria para completar el análisis principal (Simplemente sube puntos en la evaluación de la práctica).

Si ya completaste la descripción principal, compara `cnt` entre los dos grupos de `workingday`. Informa tamaño, media, mediana y desviación típica; después estima una diferencia de medias con intervalo de confianza y un contraste de dos muestras.

**Dónde mirar:** en [S2.2, comparación de medias independientes](sesion2/s2_2.ipynb), revisa el gráfico, el contraste t y la interpretación de la diferencia entre grupos. En [S2.1, intervalos y contrastes](sesion2/s2_1.ipynb), consulta cómo comunicar el intervalo y el p-valor.

Escribe la hipótesis nula, el estadístico y el p-valor. Interpreta el intervalo y el contraste por separado, distingue significación estadística de importancia práctica y recuerda que las horas de un mismo día no son independientes.

In [ ]:
# Ampliación opcional — comparación de workingday
# Referencias: S2.2, comparación de medias independientes; S2.1, intervalos y contrastes.
# Resume cnt por workingday: tamaño, media, mediana y desviación típica.
# Pista: groupby("workingday")["cnt"].agg(["count", "mean", "median", "std"]).

# TODO: define los dos grupos y calcula diferencia de medias e intervalo de Welch.
# Patrón del contraste: stats.ttest_ind(grupo_a, grupo_b, equal_var=False).
# Informa estadístico, grados de libertad y p-valor; comenta que horas del mismo día dependen entre sí.


<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 1 — Diferencia de medias</strong><br>¿Qué hipótesis contrastas y qué grupo tiene mayor media? Indica tamaños y diferencia estimada.<br><em>Pista: define laborable − no laborable; la nula plantea diferencia cero. Usa el resumen por grupo para signos y cifras.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Escribe H0/H1, tamaños de grupos, medias y diferencia (laborable menos no laborable).

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2 — Intervalo y contraste</strong><br>¿Qué intervalo de confianza y p-valor obtuviste, y cuál es la decisión al 5 %?<br><em>Pista: mira si el intervalo incluye cero y compara el p-valor con 0,05; no afirmes igualdad si no rechazas H0.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Reporta el IC, estadístico y p-valor; decide si hay evidencia de diferencia, sin interpretar no-significación como igualdad.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3 — Relevancia y límites</strong><br>¿La diferencia parece importante en alquileres por hora y qué limitación tiene el contraste?<br><em>Pista: además del p-valor, compara la diferencia con la dispersión y recuerda que horas de un mismo día pueden depender entre sí.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Comenta magnitud práctica y dependencia entre horas, además de la evidencia estadística.

</div>

## 4. Caso 1 — Bike: verificar la partición

Completa la tabla con fechas y tamaños. Después, explica brevemente por qué se separa cronológicamente, qué decisiones se toman con validación y para qué se reserva el test. La disponibilidad de pronósticos meteorológicos se comenta en el apartado del modelo múltiple.

**Referencia:** [S3, apartado 3.13, evaluar con datos nuevos](sesion3/s3.ipynb). Aquí se adapta a una serie temporal: se separan fechas completas, no filas al azar.

No uses las respuestas del test para elegir variables ni para corregir el modelo.

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí (más la tabla de abajo). Completa la tabla y explica por qué la separación cronológica es adecuada, qué se decide con validación y qué papel tiene el test. Indica si las variables meteorológicas estarían disponibles al predecir.</div>

| Bloque | Número de filas | Primera fecha | Última fecha | Uso |
|---|---:|---|---|---|
| Ajuste (`bike_ajuste`) |  |  |  | Describir, ajustar y diagnosticar |
| Validación (`bike_validacion`) |  |  |  | Elegir el modelo de Bike |
| Entrenamiento externo (`bike_train`) |  |  |  | Reajuste final |
| Test (`bike_test`) |  |  |  | Evaluación final, una sola vez |

## 5. Caso 1 — Bike: regresión lineal
### 5.1. Modelo simple

Usa **solo `bike_ajuste`** para ajustar `cnt` en función de `temp`. En el bloque de código, la respuesta y la matriz con intercepto ya están preparadas; **no cambies esas líneas**. Completa únicamente los `TODO`: ajustar OLS, mostrar el resumen y añadir el gráfico.

**Referencia:** [S3, apartado 3.3, modelo lineal simple](sesion3/s3.ipynb).


In [ ]:
# 5 — Regresión lineal en Bike
# YA PREPARADO — NO MODIFICAR: respuesta y matriz simple con intercepto.
y_bike = bike_ajuste["cnt"]

# 5.1 — Modelo simple: cnt en función de temp
X_simple = sm.add_constant(bike_ajuste[["temp"]], has_constant="add")

# TODO: ajusta modelo_simple con sm.OLS(y_bike, X_simple).fit().


# TODO: muestra modelo_simple.summary().



### 5.2. Modelo múltiple

Completa el ajuste del modelo con `temp`, `hum`, `windspeed` e indicadores de `hr`. **La lista de variables, la creación de indicadores y la alineación de las columnas de validación ya están preparadas: no las modifiques.** Completa las llamadas de ajuste y el resumen que se indican como `TODO`.

**Referencias:** [S3, apartado 3.4, modelo multivariable](sesion3/s3.ipynb) y [S4, apartado 4.2, categorías de referencia](sesion4/s4.ipynb).

Interpreta la pendiente, el intercepto y $R^2$ del modelo simple; después compara la pendiente de `temp` y un coeficiente de hora en el múltiple. Recuerda que `cnt` es un recuento discreto modelado con OLS, y que una asociación ajustada no demuestra causalidad. Para predecir horas futuras, las variables meteorológicas requieren pronósticos.

In [ ]:
# 5.2 — Modelo múltiple: variables meteorológicas y hora como categoría
# YA PREPARADO — NO MODIFICAR: variables, dummies e índices de columnas para validación.
variables_modelo = ["temp", "hum", "windspeed", "hr"]
X_multiple_sin_const = pd.get_dummies(
    bike_ajuste[variables_modelo], columns=["hr"], drop_first=True, dtype=float
)
X_multiple = sm.add_constant(X_multiple_sin_const, has_constant="add")

X_multiple_validacion_sin_const = pd.get_dummies(
    bike_validacion[variables_modelo], columns=["hr"], drop_first=True, dtype=float
).reindex(columns=X_multiple_sin_const.columns, fill_value=0)
X_multiple_validacion = sm.add_constant(
    X_multiple_validacion_sin_const, has_constant="add"
).reindex(columns=X_multiple.columns, fill_value=0)
X_simple_validacion = sm.add_constant(
    bike_validacion[["temp"]], has_constant="add"
).reindex(columns=X_simple.columns, fill_value=0)

# TODO: ajusta modelo_multiple con sm.OLS(y_bike, X_multiple).fit().


# TODO: muestra el resumen y compara params["temp"] con el del modelo simple.


# PISTA: Pista: modelo_multiple = sm.OLS(..., ...).fit(); después comparo
# modelo_simple.params[...] con modelo_multiple.params[...].
# Referencia: sesion3/s3.ipynb, celdas 35–37 (modelo lineal multivariable).

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 1 — Modelo lineal simple</strong><br>¿Qué predice el modelo y cómo interpretas la pendiente, el intercepto y el <em>R²</em>?<br><em>Pista: cita coeficiente, signo y unidad; el intercepto corresponde a <code>temp=0</code> y R² resume ajuste, no aciertos.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Identifica respuesta/predictor y cita pendiente con unidad, intercepto y significado de R².

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2 — Modelo múltiple</strong><br>¿Qué compara un coeficiente categórico y cuál es su referencia? ¿Cambió el coeficiente de <code>temp</code> frente al modelo simple?<br><em>Pista: <code>drop_first=True</code> deja una categoría base; compara los dos valores de <code>params["temp"]</code>.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Nombra una categoría de referencia, interpreta un coeficiente de dummy y compara las pendientes de <code>temp</code>.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3 — Interpretación y límites</strong><br>¿Por qué una asociación ajustada no demuestra causalidad?<br><em>Pista: considera que el modelo solo controla las variables incluidas y que los datos no provienen de una asignación experimental.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

## 6. Caso 1 — Bike: revisar residuos

Diagnostica el modelo múltiple ajustado solo con `bike_ajuste` mediante dos gráficos básicos:

1. Residuos frente a valores ajustados.
2. Gráfico Q–Q de residuos.

**Ya están calculados en la celda** los residuos y los valores ajustados del modelo; no vuelvas a calcularlos ni cambies el modelo. Completa solo las instrucciones `TODO` para dibujar los dos gráficos.

**Referencia:** [S3, apartado 3.6, diagnóstico del modelo lineal](sesion3/s3.ipynb). No hace falta calcular distancias de Cook ni identificar observaciones influyentes en esta versión reducida.

Interpreta los gráficos en las preguntas siguientes.

In [ ]:
# 6 — Diagnósticos básicos del modelo múltiple (solo bike_ajuste)
# Referencia: sesion3/s3.ipynb, celdas 49–51 (diagnóstico con residuos y gráficos).
# YA PREPARADO — NO MODIFICAR: residuos y valores ajustados del modelo.
if "modelo_multiple" not in globals():
    raise ValueError("No has definido modelo_multiple en el punto 5.2")
residuos = modelo_multiple.resid
ajustados = modelo_multiple.fittedvalues

# 1. Residuos frente a valores ajustados
fig, ax = plt.subplots(figsize=(7, 4))
# TODO: añade los puntos con ax.scatter(ajustados, residuos, alpha=0.25).


# TODO: añade la línea horizontal en cero, etiquetas, título y plt.show().



# 2. Gráfico Q–Q
# TODO: aplica sm.qqplot(residuos, line="45"), añade título y muestra la figura.


# Comenta qué patrones ves; el gráfico no demuestra por sí solo todos los supuestos.



<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 1 — Residuos</strong><br>¿Los residuos se centran en cero? Se aprecia curvatura o cambia su dispersión con los ajustados? <br> ¿Qué intenamos analizar con esto y qué consencuencias tiene que no se cumplan estas condiciones?<br><em>Pista: busca un patrón en el primer gráfico; una forma de abanico sugiere variabilidad no constante.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2 — Normalidad aproximada</strong><br>¿Qué muestra el gráfico Q–Q sobre la distribución de los residuos? <br> ¿Qué intenamos analizar con esto y qué consencuencias tiene que no se cumplan estas condiciones? <br><em>Pista: describe si los puntos siguen la recta y qué ocurre en los extremos; es una revisión visual, no una prueba definitiva.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3 — Limitación del modelo</strong><br>¿Qué limitación del modelo te preocupa más a partir de los residuos y por qué el diagnóstico visual no basta para decidir que el modelo es válido?<br><em>Pista: considera curvatura, dispersión, forma de los residuos y la diferencia entre diagnóstico y rendimiento en test.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Menciona un patrón residual o supuesto que te preocupe y explica por qué hace falta evaluar en test separado.

</div>

## 7. Caso 1 — Bike: comparar, validar y evaluar

### 7.1. Comparar dos modelos en el bloque de ajuste

Compara, con las mismas filas de `bike_ajuste`, el modelo simple (`temp`) y el múltiple (`temp`, `hum`, `windspeed` e indicadores de `hr`). Registra sus AIC. El AIC compara el equilibrio entre ajuste y complejidad; no estima el error futuro.

**Referencia:** [S3, apartados 3.13–3.14](sesion3/s3.ipynb). No uses el test para cambiar el modelo.

In [ ]:
# 7.1 — AIC de los dos modelos (ajustados con las mismas filas)
# TODO: recupera modelo_simple.aic y modelo_multiple.aic en una tabla.


# Pistas: pd.DataFrame, nombres de fila "Simple" y "Múltiple", display(...).
# Pista (completa los huecos):
# tabla_aic = pd.Series({"Simple": ..., "Múltiple": ...})
# Referencia: sesion3/s3.ipynb, celdas 42–44 (criterio y selección por AIC).

### 7.2. Elegir con la validación cronológica

Calcula RMSE para ambos modelos en `bike_validacion` y elige según error y sencillez. Esta es una sola ventana temporal, no validación cruzada. No reajustes con validación ni uses test para elegir.

**Referencia:** [S3, apartados 3.13–3.14](sesion3/s3.ipynb). No uses el test para cambiar el modelo.

In [ ]:
# 7.2 — RMSE de validación; no reajustes los modelos con bike_validacion
# TODO: predice con modelo_simple/X_simple_validacion y modelo_multiple/X_multiple_validacion.


# TODO: calcula RMSE de ambos con np.sqrt(mean_squared_error(y_validacion, prediccion)).


# TODO: muestra una serie o tabla comparativa y elige según RMSE y sencillez.


# Pistas: predigo ambos modelos en validación y calculo
# rmse = np.sqrt(mean_squared_error(..., ...)); comparo los dos resultados.
# IMPORTANTE: modelo_bike_elegido = ...  # escribe "simple" o "multiple" según validación
# La validación no se usa para reajustar ni el test para elegir.

### 7.3. Evaluación final

Reajusta el elegido con todo `bike_train` y evalúalo una sola vez en `bike_test`. Calcula MAE, RMSE y $R^2$, compara observado con predicho y revisa los errores absolutos mayores. Los errores se expresan en alquileres por hora.

**Partes ya preparadas en el bloque de código — no modificar:** la rama que construye y alinea los diseños de entrenamiento/test según el modelo elegido. Completa los `TODO` para crear la tabla AIC, comparar RMSE, escribir el nombre del modelo elegido y ajustar/evaluar el modelo final.

**Referencia:** [S3, apartados 3.13–3.14](sesion3/s3.ipynb). No uses el test para cambiar el modelo.

In [ ]:
# 7.3 — YA PREPARADO: diseño final y alineamiento entre bike_train y bike_test.
# No cambies esta rama; depende de modelo_bike_elegido, que debes decidir arriba.
if "modelo_bike_elegido" not in globals():
    raise ValueError("No has definido model_bike_elegido en el punto 7.2")
if modelo_bike_elegido == "simple":
    X_bike_train_final = sm.add_constant(bike_train[["temp"]], has_constant="add")
    X_bike_test_final = sm.add_constant(bike_test[["temp"]], has_constant="add")
else:
    X_bike_train_sin_const = pd.get_dummies(
        bike_train[variables_modelo], columns=["hr"], drop_first=True, dtype=float
    )
    X_bike_test_sin_const = pd.get_dummies(
        bike_test[variables_modelo], columns=["hr"], drop_first=True, dtype=float
    ).reindex(columns=X_bike_train_sin_const.columns, fill_value=0)
    X_bike_train_final = sm.add_constant(X_bike_train_sin_const, has_constant="add")
    X_bike_test_final = sm.add_constant(X_bike_test_sin_const, has_constant="add").reindex(
        columns=X_bike_train_final.columns, fill_value=0
    )

# TODO: ajusta modelo_bike_final con sm.OLS(bike_train["cnt"], X_bike_train_final).fit().


# TODO: predice bike_test; calcula MAE, RMSE y R² con las funciones ya importadas.


# TODO: muestra observado frente a predicho y revisa los diez errores absolutos mayores.


# Pista: ajusto con bike_train, predigo sobre X_bike_test_final y comparo
# con bike_test["cnt"]; después resumo errores. No cambio el modelo tras ver test.
# Referencia: sesion3/s3.ipynb, celdas 125–130 (reservar y predecir en test)
# y 134–140 (RMSE, MAE y resumen de métricas).

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 1 — AIC y modelos candidatos</strong><br>¿Qué modelo tiene menor AIC? ¿Se comparó la misma respuesta y las mismas filas, y qué penaliza el AIC?<br><em>Pista: el AIC equilibra ajuste y complejidad; no mide directamente el error futuro.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Nombra el modelo con menor AIC, confirma filas/respuesta y señala que penaliza la complejidad.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2 — Selección mediante validación</strong><br>¿Qué RMSE obtuvo cada candidato en validación, cuál eliges y qué límite tiene esa elección?<br><em>Pista: elige el menor RMSE y recuerda que es una sola ventana cronológica, no una validación cruzada.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Cita los RMSE de validación, di cuál escogiste y menciona la limitación de usar una sola ventana temporal.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3 — Evaluación final de Bike</strong><br>¿Qué MAE, RMSE y R² obtuvo el modelo en test, dónde falla más y qué permite concluir sobre su predicción?<br><em>Pista: expresa MAE/RMSE en alquileres por hora; usa la tabla de mayores errores y no vuelvas a elegir el modelo con test.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Incluye las tres métricas con unidades, menciona el patrón de error y limita la conclusión al periodo de test.

</div>

## 8. Caso 2 — Diabetes: preparar y describir los datos

Antes de crear clases, describe `target` en su escala continua. Ejecuta primero la celda de carga de 8.1 y luego completa el análisis de 8.2, sin modificar la respuesta.

**Referencias:** [S1, descriptiva univariante y bivariante](sesion1/s1.ipynb) y [S3, apartados 3.1–3.2](sesion3/s3.ipynb).

### 8.1. Cargar Diabetes

Ejecuta la celda siguiente para cargar el conjunto desde scikit-learn. Después, continúa con el apartado **8.2**, que indica qué resumen y gráficos debes preparar antes de definir las clases.

**Referencia:** la llamada `load_diabetes(as_frame=True)` está explicada en la [documentación oficial de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html).

In [ ]:
# load_diabetes se documenta en el enlace del apartado 8.1; devuelve X e información del conjunto.
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
diabetes_df = diabetes.frame.copy()

print(f"Diabetes: {diabetes_df.shape[0]} filas × {diabetes_df.shape[1]} columnas")
display(diabetes_df.head())

### 8.2. Describir `target` antes de clasificar

**Completa el bloque de código 8.2 que aparece a continuación** con `diabetes_df`:

1. Resume `target` con tamaño, media, mediana, desviación típica, cuartiles, mínimo y máximo.
2. Dibuja su histograma.
3. Dibuja `target` frente a `bmi` y calcula correlaciones entre `target` y las características numéricas.

**Dónde mirar:** para el resumen, los histogramas y la correlación, revisa [S1, descriptiva univariante y bivariante](sesion1/s1.ipynb); para el diagrama de dispersión entre respuesta y predictor, consulta [S3, exploración gráfica (3.2)](sesion3/s3.ipynb).

Escribe qué rango y dispersión observas, si la distribución parece simétrica y qué predictor tiene mayor correlación lineal. Cita la evidencia y añade dos cautelas: correlación no implica causalidad, y las características están transformadas/estandarizadas, por lo que no deben leerse como medidas clínicas originales.

In [ ]:
# 8.2 — Describir target antes de crear clases
# Referencias: sesion1/s1.ipynb, celdas 132–156 (resumen, caja e histograma),
# y celdas 180–196 (correlación y dispersión); sesion3/s3.ipynb, celdas 8–14.
target_diabetes = diabetes_df["target"]

# TODO: usa describe() y añade o identifica la mediana.


# TODO: dibuja histograma de target (pista: sns.histplot).


# TODO: dibuja target frente a bmi (pista: plt.scatter).


# TODO: calcula correlaciones entre target y las características numéricas.


# Mantén target continuo: todavía no dividas ni crees etiquetas.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 1 — Descripción de Diabetes</strong><br>¿Qué rango, centro y dispersión tiene <code>target</code> y qué sugiere la forma de su distribución?<br><em>Pista: cita mínimo, máximo, media, mediana y desviación típica; compara media-mediana con la cola del histograma.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2 — Asociación con características</strong><br>¿Qué predictor tiene mayor correlación con <code>target</code> y qué patrón ves frente a <code>bmi</code>?<br><em>Pista: ordena correlaciones por magnitud absoluta y describe dirección y dispersión del diagrama, sin inferir causa.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3 — Escala y simplificación</strong><br>¿Por qué las características no se interpretan como medidas clínicas originales y qué información se pierde al cortar <code>target</code> en dos?<br><em>Pista: considera que las variables están transformadas y que valores distintos dentro de una misma clase se vuelven indistinguibles.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.
Menciona la transformación de las características y que la dicotomización elimina diferencias de magnitud dentro de cada clase.

</div>

## 9. Caso 2 — Diabetes: crear clases y ajustar regresión logística

### 9.1. Definir la respuesta

La partición externa **ya está preparada en la celda de código**: `X_train`, `X_test`, `target_train` y `target_test` ya se separan usando la semilla indicada; la mediana de entrenamiento y las etiquetas `y_train`/`y_test` también se calculan allí. **No cambies esas líneas ni recalcules el corte.** Revisa los recuentos que imprime la celda y continúa con 9.2.

### 9.2. Ajustar los modelos

La partición interna de ajuste/validación y las matrices de diseño con constante (`X_simple_ajuste`, `X_multiple_ajuste`) también están preparadas; **no las modifiques**. Completa solo las llamadas `TODO` para ajustar dos GLM binomiales en ajuste: uno con `bmi` y otro con todas las características. Guarda los resultados como `mod_simple` y `mod_multiple`.

**Referencias:** [S4, apartado 4.2, categorías de referencia](sesion4/s4.ipynb) y [S4, apartado 4.3, GLM binomial](sesion4/s4.ipynb). Como la respuesta ya es 0/1, no copies la codificación `yes/no` del ejemplo del banco.

**Para interpretar:** consulta [S4, apartados 4.3–4.6](sesion4/s4.ipynb). El evento es clase 1. Un coeficiente positivo eleva los log-odds y su exponencial es un OR, no un aumento fijo de probabilidad. Compara AIC solo con la misma respuesta y las mismas filas.

In [ ]:
# 9.1 — Partición externa y corte de clases
# YA PREPARADO — NO MODIFICAR: partición, mediana aprendida en train y etiquetas.
X_diabetes = diabetes_df.drop(columns="target")
target_original = diabetes_df["target"]
X_train, X_test, target_train, target_test = train_test_split(
    X_diabetes, target_original, test_size=0.20, random_state=SEMILLA
)
corte = target_train.median()
y_train = (target_train > corte).astype(int)
y_test = (target_test > corte).astype(int)
print(f"Corte (mediana de entrenamiento): {corte:.2f}")
print("Clases en entrenamiento:")
display(y_train.value_counts().sort_index())
print("Clases en test:")
display(y_test.value_counts().sort_index())

# 9.2 — YA PREPARADO — NO MODIFICAR: partición interna y matrices con constante.
X_ajuste, X_validacion, y_ajuste, y_validacion = train_test_split(
    X_train, y_train, test_size=0.25, random_state=SEMILLA, stratify=y_train
)
X_simple_ajuste = sm.add_constant(X_ajuste[["bmi"]], has_constant="add")
X_multiple_ajuste = sm.add_constant(X_ajuste, has_constant="add")

# Completa solo las llamadas TODO; el test no se usa para ajustar ni elegir.
# TODO: ajusta mod_simple = sm.GLM(y_ajuste, X_simple_ajuste, family=sm.families.Binomial()).fit()


# TODO: ajusta mod_multiple con X_multiple_ajuste y la misma familia binomial.


# Pista (completa los huecos):
# mod_simple = sm.GLM(..., ..., family=...).fit()
# mod_multiple = sm.GLM(..., ..., family=...).fit()
# Referencia: sesion4/s4.ipynb, celdas 36–39 (GLM binomial y ajuste).

# TODO: muestra los dos AIC para comparar los modelos sobre los mismos casos.


# TODO: calcula OR para bmi como np.exp(mod_multiple.params["bmi"]).


# Pista: aic_simple = ...; aic_multiple = ...; OR = np.exp(...).
# Referencia para odds ratio: sesion4/s4.ipynb, celdas 65–68.
# Interpreta el OR en odds, no como un cambio directo de probabilidad.
# El evento positivo es la clase 1: target por encima de la mediana de entrenamiento.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"> Completa la tabla con AIC y el coeficiente/OR elegido para cada modelo: </div>



| Modelo | Entradas | AIC (ajuste) | Coeficiente elegido | OR e interpretación |
|---|---|---:|---|---|
| Simple | `bmi` |  |  |  |
| Múltiple | Todas |  |  |  |

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 1 — Definición de clase</strong><br>¿Qué significa clase 1, con qué corte se definió y por qué no es un diagnóstico?<br><em>Pista: la mediana se calcula solo con <code>target_train</code>; aplica el mismo umbral a test.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respueta aquí. Describe la clase positiva, el corte obtenido solo con train y el límite de interpretación clínica.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2 — Coeficientes logísticos</strong><br>¿Cómo interpretas el signo, el odds ratio y el intercepto del modelo múltiple?<br><em>Pista: <code>exp(β)</code> multiplica odds por unidad (o por 0,1 si así lo escalas); el intercepto corresponde a entradas iguales a cero.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Elige un coeficiente, explica dirección y OR, y describe el perfil que implica el intercepto.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3 — Comparación de modelos</strong><br>¿Qué modelo tiene menor AIC y qué permite —y qué no permite— concluir esa comparación?<br><em>Pista: compara AIC solo con la misma respuesta y las mismas filas; menor AIC favorece ajuste-complejidad, no demuestra mejor rendimiento futuro ni causalidad.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu repuesta aquí.

</div>

## 10. Caso 2 — comparar, fijar la regla y evaluar

Usa los modelos ajustados en el apartado anterior. **La constante y el umbral didáctico 0,5 ya están especificados en la celda de código; no los cambies.** Completa únicamente los cálculos `TODO` y escribe las respuestas.

### 10.1. Comparar modelos en validación

1. Las matrices de validación con constante ya están construidas en la celda de código inferior; no las vuelvas a definir ni las modifiques. Calcula las probabilidades de clase 1 para ambas usando los modelos ajustados, sin volver a ajustarlos.
2. Calcula ROC-AUC para ambos modelos y completa la tabla. El AUC evalúa el orden de las puntuaciones, no el porcentaje de aciertos.
3. Prefiere el modelo con mayor AUC; si los AUC son iguales, elige el más sencillo. Justifica la decisión antes de abrir el test.

**Dónde mirar:** [S4, apartados 4.7 y 4.9](sesion4/s4.ipynb) muestran cómo obtener probabilidades y ROC-AUC; [S4, apartado 4.10](sesion4/s4.ipynb) explica las métricas y sus denominadores. En la celda de código inferior, `X_valid_simple` y `X_valid_multiple` ya quedan definidas con las columnas necesarias y la constante. Completa allí solo las predicciones, los AUC y la selección del modelo.

In [ ]:
# 10.1 — Probabilidades y ROC-AUC en validación
# YA PREPARADO — NO MODIFICAR: matrices de validación con constante y nombres de columnas.
X_valid_simple = sm.add_constant(X_validacion[["bmi"]], has_constant="add")
X_valid_multiple = sm.add_constant(X_validacion, has_constant="add")
# Completa solo las llamadas TODO; no vuelvas a ajustar los modelos.
# TODO: predice con mod_simple y mod_multiple; guarda p_valid_simple y p_valid_multiple.


# TODO: calcula sus AUC con roc_auc_score(y_validacion, probabilidades).


# TODO: elige "simple" o "multiple" según AUC y sencillez; guarda p_valid_elegido.
# IMPORTANTE: Guarda el nombre en modelo_elegido. Es decir: modelo_elegido = ...


# Pista (completa los huecos):
# p_valid_simple = mod_simple.predict(...)
# auc_simple = roc_auc_score(..., ...)
# Después repito para el múltiple y conservo el vector de probabilidades elegido.
# Referencia: sesion4/s4.ipynb, celdas 85–87 (probabilidades y ROC-AUC).

### 10.2. Evaluar con un umbral fijado de antemano

El umbral **0,5 ya está fijado**. Predice clase 1 si la probabilidad es mayor que 0,5; si es igual o menor, predice clase 0. No busques ni optimices otro umbral.

Con el modelo elegido, calcula sensibilidad, especificidad y precisión en validación con ese umbral. La orientación de la matriz es **predicción en filas y clase real en columnas**; aplica la regla a `pd.crosstab(prediccion, realidad)`. Desde sus cuatro celdas, sensibilidad = TP/(TP+FN), especificidad = TN/(TN+FP) y precisión = TP/(TP+FP). Para F1 puedes usar `f1_score`.

**Dónde mirar:** [S4, apartado 4.8, umbrales](sesion4/s4.ipynb) explica la regla de clasificación y [S4, apartado 4.10](sesion4/s4.ipynb) muestra la matriz y los denominadores.

Explica qué significa cada una de las tasas y qué limitación tiene evaluar con una sola regla fija.

In [ ]:
# 10.2 — YA PREPARADO — NO MODIFICAR: umbral fijado de antemano en 0,5.
UMBRAL_FIJADO = 0.5
# TODO: clasifica como 1 si p_valid_elegido > UMBRAL_FIJADO; si no, clase 0.


# TODO: construye la tabla con pd.crosstab(pred_validacion, y_validacion).
# Desde TN, FP, FN y TP, calcula sensibilidad, especificidad y precisión.


# Pista: pred_validacion = (... > UMBRAL_FIJADO).astype(int)
# matriz = pd.crosstab(..., ...); luego leo sus cuatro celdas y calculo las tasas.
# Referencia: sesion4/s4.ipynb, celdas 94–95 (regla del umbral) y 119–120 (matriz).
# Pistas: confusion_matrix(..., labels=[0, 1]).ravel() o lee las celdas del crosstab.


### 10.3. Evaluación final en test

**El diseño final para el modelo simple o múltiple ya está preparado en el bloque de código siguiente; no cambies esa rama ni la selección realizada con validación.** Completa los `TODO`: ajustar el modelo elegido con todo el entrenamiento externo, predecir en test y calcular las métricas con el mismo umbral de 0,5.

El test no se usa para modificar el modelo ni el umbral. La evaluación describe este conjunto, no garantiza el rendimiento en otra muestra.


In [ ]:
# 10.3 — YA PREPARADO — NO MODIFICAR: diseños finales, con las columnas necesarias.
if "modelo_elegido" not in globals():
    raise ValueError("No has definido model_elegido en el punto 10.1")
elif modelo_elegido == "simple":
    X_train_final = sm.add_constant(X_train[["bmi"]], has_constant="add")
    X_test_final = sm.add_constant(X_test[["bmi"]], has_constant="add")
else:
    X_train_final = sm.add_constant(X_train, has_constant="add")
    X_test_final = sm.add_constant(X_test, has_constant="add")
# Completa los TODO: ajusta con todo X_train/y_train y evalúa test una sola vez.
# TODO: ajusta modelo_final con todo X_train/y_train usando GLM binomial.


# TODO: predice p_test y aplica el mismo UMBRAL_FIJADO; no vuelvas a elegirlo.


# TODO: calcula la matriz de confusión, sensibilidad, especificidad, precisión,
#       F1 y ROC-AUC. El test es evaluación final: no modifiques el modelo.


# Pista: modelo_final = ...fit(); p_test = modelo_final.predict(...)
# y aplico el mismo umbral, una sola vez, sin volver a elegir el modelo.
# Referencia: sesion4/s4.ipynb, celdas 85–87 (AUC), 94–95 (umbral)
# y 119–120 (matriz de confusión y tasas).

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 1 — Comparar modelos</strong><br>¿Qué ROC-AUC obtuvo cada modelo en validación y cuál eliges?<br><em>Pista: el AUC se calcula con probabilidades; compara los valores y considera también la sencillez.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí. Anota los dos AUC y justifica brevemente el modelo elegido antes de abrir test.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2 — Validación</strong><br>Con el umbral 0,5, ¿qué sensibilidad, especificidad y precisión obtuviste en validación? Explica los denominadores.<br><em>Pista: sensibilidad usa positivos reales; especificidad, negativos reales; precisión, predicciones positivas.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3 — Test y matriz de confusión</strong><br>¿Qué clase es positiva, cuáles son TP/FP/TN/FN y qué tasas finales obtuviste?<br><em>Pista: la práctica define clase 1 como positiva. Con predicción en filas y realidad en columnas: sensibilidad usa TP/(TP+FN), especificidad TN/(TN+FP), precisión TP/(TP+FP).</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br>

Tu respuesta aquí.

</div>

## 11. Revisión final de la práctica

No se solicita un informe global adicional ni una extensión determinada. Entrega las respuestas a las preguntas de la plantilla y revisa esta lista antes de finalizar:

- [ ] Excluí `casual` y `registered` de las predictoras de `cnt`.
- [ ] Separé Bike cronológicamente por fechas completas y usé ajuste para explorar y entrenar, validación para elegir y test solo para evaluar al final.
- [ ] Tuve en cuenta que `cnt` es un recuento discreto y que los predictores meteorológicos futuros requieren pronósticos.
- [ ] Calculé la mediana de `target` solo con entrenamiento y apliqué ese corte a test.
- [ ] Elegí el modelo con validación y mantuve el umbral 0,5 fijado de antemano.
- [ ] Interpreté cada métrica con unidades o denominador.
- [ ] No presenté asociación como causalidad ni la clase de Diabetes como diagnóstico.
- [ ] Reinicié el kernel y comprobé que el cuaderno se ejecuta en orden.